# Sanity-check графовых признаков

Анализ распределений и проверка на аномалии для фичей:
- `msg_sim_degree` — степень связности сообщения
- `ch_out_degree` — исходящая степень канала
- `ch_pagerank` — PageRank канала
- `msg_clustering` — коэффициент кластеризации
- `nbr_hist_viral_share` — доля исторически вирусных соседей

In [1]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
ART_DIR = ROOT / "artifacts"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

DATA_DIR = str(ROOT / "data")
ART_DIR = str(ROOT / "artifacts")

FEATURES = ['msg_sim_degree', 'ch_out_degree', 'ch_pagerank', 'msg_clustering', 'nbr_hist_viral_share']


In [5]:
!pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# Загрузка graph_features.csv из artifacts
path = os.path.join(ART_DIR, "graph_features.csv")
df = pd.read_csv(path)
print(f"Загружен: {path}")
print(f"Строк: {len(df):,}")
df.head()

Загружен: ./data\Датасет_для_разметки_filled_20260220_021731.xlsx
Строк: 500


,message,topic,LLm (1),Экономический эффект (1),Информационный резонанс (1),Правильность определения темы (1),Экономический нарратив (1),Сила нарратива (1),Комментарий (1),LLm (2),...,Сила нарратива (5),Комментарий (5),message_norm,id,id_channel,msg_sim_degree,ch_out_degree,ch_pagerank,msg_clustering,nbr_hist_viral_share
0,#POSI \nГК Позитив — лидер в рейтинге покупок ...,Рынки капитала,gpt-oss:20b-cloud,0.0,1.0,3.0,Нет,1.0,Новость о лидирующей покупке акций конкретной ...,gpt-oss:20b-cloud,...,1,Новость представляет собой статистический отчё...,#POSI \nГК Позитив — лидер в рейтинге покупок ...,8c497c96-be46-4bb4-888f-25e40d20debb,2,0,11724,0.043697,0.0,0.0
1,⚡️Россельхознадзор опроверг запрет на ввоз ман...,Международная торговля,gpt-oss:20b-cloud,0.0,1.0,1.0,Нет,1.0,Новость представляет собой простое опровержени...,gpt-oss:20b-cloud,...,1,"Новость представляет собой разъяснение, а не с...",⚡️Россельхознадзор опроверг запрет на ввоз ман...,63c159f3-4205-4477-bb06-e00c0bde9187,1,0,23522,0.087665,0.0,0.0
2,Корпоративные бонды с доходностью 20-21%+ — се...,Рынки капитала,gpt-oss:20b-cloud,0.0,1.0,1.0,Нет,1.0,Новость представляет собой простое перечислени...,gpt-oss:20b-cloud,...,1,Новость представляет собой техническую информа...,Корпоративные бонды с доходностью 20-21%+ — се...,9d73c104-aa5e-478d-bc27-d57f15c0c646,2,0,11724,0.043697,0.0,0.0
3,#MTLR\n🪨 Мечел в 2025г может сократить отгрузк...,Корпоративные финансы,gpt-oss:20b-cloud,-1.0,1.0,1.0,Нет,1.0,Новость фокусируется на корпоративных решениях...,gpt-oss:20b-cloud,...,1,Новость представляет собой корпоративный финан...,#MTLR\n🪨 Мечел в 2025г может сократить отгрузк...,db2e74c1-fefa-4333-823c-6e0a527e87a0,2,1,11724,0.043697,0.0,0.0
4,Россия вошла в топ-5 стран по оттоку миллионер...,Макроэкономика,gpt-oss:20b-cloud,-1.0,1.0,3.0,Нет,1.0,Новость представляет собой статистический факт...,gpt-oss:20b-cloud,...,1,Новость представляет собой статистический факт...,Россия вошла в топ-5 стран по оттоку миллионер...,031d93c6-89f0-426a-ada0-9f6f4174f4b7,18,0,12549,0.046771,0.0,0.0


In [7]:
# Проверка наличия колонок
missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print("Доступные колонки:", list(df.columns))
    raise ValueError(f"Отсутствуют колонки: {missing}")
print("Все фичи найдены.")

Все фичи найдены.


## 1. Базовая статистика и пропуски

In [8]:
stats_df = df[FEATURES].describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
stats_df['missing'] = df[FEATURES].isna().sum()
stats_df['missing_pct'] = (stats_df['missing'] / len(df) * 100).round(2)
display(stats_df)

,count,mean,std,min,25%,50%,75%,max,missing,missing_pct
msg_sim_degree,500.0,0.756000,1.523518,0.000000,0.000000,0.000000,1.000000,20.000000,0,0.0
ch_out_degree,500.0,15361.438000,7986.067183,7917.000000,11724.000000,12549.000000,14219.000000,39739.000000,0,0.0
ch_pagerank,500.0,0.057252,0.029762,0.029509,0.043697,0.046771,0.052995,0.148101,0,0.0
msg_clustering,500.0,0.066561,0.228642,0.000000,0.000000,0.000000,0.000000,1.000000,0,0.0
nbr_hist_viral_share,500.0,0.015333,0.115530,0.000000,0.000000,0.000000,0.000000,1.000000,0,0.0


## 2. Проверка на аномалии

In [ ]:
print("=== Отрицательные значения (недопустимы для этих фичей) ===")
for col in FEATURES:
    neg = (df[col] < 0).sum()
    print(f"{col}: {neg}")

print("\n=== Значения вне ожидаемых границ ===")
# ch_pagerank и msg_clustering, nbr_hist_viral_share — в [0, 1]
for col in ['ch_pagerank', 'msg_clustering', 'nbr_hist_viral_share']:
    if col in df.columns:
        out = ((df[col] < 0) | (df[col] > 1)).sum()
        print(f"{col} вне [0,1]: {out}")

# msg_sim_degree, ch_out_degree — неотрицательные целые (или float)
for col in ['msg_sim_degree', 'ch_out_degree']:
    if col in df.columns:
        inf_cnt = np.isinf(df[col]).sum()
        nan_cnt = df[col].isna().sum()
        print(f"{col} inf/nan: {inf_cnt}/{nan_cnt}")

## 3. Тесты на нормальность (Shapiro-Wilk, D'Agostino)

In [ ]:
norm_results = []
for col in FEATURES:
    x = df[col].dropna()
    if len(x) > 5000:
        x = x.sample(5000, random_state=42)  # Shapiro ограничен ~5000
    stat_sw, p_sw = stats.shapiro(x)
    stat_da, p_da = stats.normaltest(x)
    norm_results.append({
        'feature': col,
        'shapiro_p': p_sw,
        'dagostino_p': p_da,
        'normal_sw': p_sw > 0.05,
        'normal_da': p_da > 0.05
    })

norm_df = pd.DataFrame(norm_results)
display(norm_df)
print("\nНормальность: p > 0.05 → распределение не отвергается как нормальное.")

## 4. Распределения (гистограммы)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(FEATURES):
    ax = axes[i]
    data = df[col].dropna()
    ax.hist(data, bins=50, edgecolor='white', alpha=0.8)
    ax.set_title(col)
    ax.set_xlabel('')

axes[-1].axis('off')
plt.suptitle('Распределения графовых признаков', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Логарифмические шкалы для тяжёлых хвостов

In [ ]:
# msg_sim_degree и ch_out_degree часто имеют тяжёлые правые хвосты
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ['msg_sim_degree', 'ch_out_degree']):
    x = df[col].replace(0, 0.5)  # log(0) → замена на 0.5
    ax.hist(np.log1p(x), bins=50, edgecolor='white', alpha=0.8)
    ax.set_title(f'log1p({col})')

plt.suptitle('Лог-трансформация для счётных признаков')
plt.tight_layout()
plt.show()

## 6. Доли нулей и выбросы (IQR)

In [ ]:
print("=== Доля нулей ===")
for col in FEATURES:
    zero_pct = (df[col] == 0).mean() * 100
    print(f"{col}: {zero_pct:.1f}%")

print("\n=== Выбросы (IQR: за пределами Q1-1.5*IQR, Q3+1.5*IQR) ===")
for col in FEATURES:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
    out = ((df[col] < low) | (df[col] > high)).sum()
    print(f"{col}: {out} ({out/len(df)*100:.1f}%)")

## 7. Корреляции

In [ ]:
corr = df[FEATURES].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True)
plt.title('Корреляции графовых признаков')
plt.tight_layout()
plt.show()

## 8. Итоговый отчёт

In [ ]:
print("SANITY-CHECK SUMMARY")
print("="*50)
print(f"Датасет: {len(df):,} строк")
print(f"Пропуски: {df[FEATURES].isna().sum().sum()} всего")
print(f"Отрицательные: {(df[FEATURES] < 0).any(axis=1).sum()} строк")
print(f"Нормальность: большинство графовых фичей не нормальны (ожидаемо)")
print(f"Рекомендация: при моделировании учитывать тяжёлые хвосты (log-трансформация).")